# Analysis

## Loading data

In [1]:
import os
import pandas as pd
import clarite as cl
import numpy as np
import scipy.stats as stats

rpy2 ModuleSpec(name='rpy2', loader=<_frozen_importlib_external.SourceFileLoader object at 0x7ff00845fa10>, origin='/home/tomas/anaconda3/envs/py_clarite/lib/python3.7/site-packages/rpy2/__init__.py', submodule_search_locations=['/home/tomas/anaconda3/envs/py_clarite/lib/python3.7/site-packages/rpy2'])


In [2]:
#### SET PATHS
os.chdir('..')
mainpath = os.getcwd()
datapath = os.path.join(mainpath, 'Data')
respath  = os.path.join(mainpath, 'Results')

In [3]:
os.chdir(respath)
loadfiles = ['Discovery_Females', 'Discovery_Males', 'Replication_Females', 'Replication_Males']
dfs = []
for i in range(4):
    dfs.append(pd.read_csv('CleanData_' + loadfiles[i] + '.csv').set_index('ID'))

os.chdir(os.path.join(datapath, 'nh_99-06'))
var_description = pd.read_csv('VarDescription.csv')\
                     .drop_duplicates()\
                     .set_index('var')

# Convert variable descriptions to a dictionary for convenience
var_descr_dict = var_description['var_desc'].to_dict()

## Categorize variables

In [4]:
for i in range(len(dfs)):
    dfs[i] = cl.modify.categorize(dfs[i])

Running categorize
--------------------------------------------------------------------------------
0 of 390 variables (0.00%) are classified as constant (1 unique value).
44 of 390 variables (11.28%) are classified as binary (2 unique values).
5 of 390 variables (1.28%) are classified as categorical (3 to 6 unique values).
325 of 390 variables (83.33%) are classified as continuous (>= 15 unique values).
0 of 390 variables (0.00%) were dropped.
16 of 390 variables (4.10%) were not categorized and need to be set manually.
	16 variables had between 6 and 15 unique values
	0 variables had >= 15 values but couldn't be converted to continuous (numeric) values
Running categorize
--------------------------------------------------------------------------------
0 of 390 variables (0.00%) are classified as constant (1 unique value).
44 of 390 variables (11.28%) are classified as binary (2 unique values).
5 of 390 variables (1.28%) are classified as categorical (3 to 6 unique values).
327 of 390 

In [5]:
text  = ['Discovery females', 'Discovery males', 'Replication females', 'Replication males']
for i in range(len(dfs)):
    var_types   = cl.describe.get_types(dfs[i])
    var_unknown = var_types[var_types == 'unknown'].index
    print('Unknown variables in ' + text[i] + ':')
    for v in list(var_unknown):
        if v in var_descr_dict:
            print(f"\t{v} = {var_descr_dict[v]}")

Unknown variables in Discovery females:
	LBXALD = Aldrin (ng/g)
	DRD350BQ = # of times crabs eaten in past 30 days
	LBX151 = PCB151 (ng/g)
	DRD350HQ = # of times shrimp eaten in past 30 days
	DRD370DQ = # of times catfish eaten in past 30 days
	LBX149 = PCB149 (ng/g)
	LBX087 = PCB87 (ng/g)
	URXUPT = Platinum, urine (ng/mL)
	LBDBANO = Basophils number
	DRD370MQ = # of times salmon eaten in past 30 days
	DRD370AQ = # of times breaded fish products eaten
	LBX189 = PCB189 (ng/g)
	URXUBE = Beryllium, urine (ng/mL)
	LBXEND = Endrin (ng/g)
	DRD370TQ = # of times other fish eaten past 30 days
	quantity_drink_per_day = drink per day
Unknown variables in Discovery males:
	LBXALD = Aldrin (ng/g)
	DRD350BQ = # of times crabs eaten in past 30 days
	DRD370DQ = # of times catfish eaten in past 30 days
	LBX149 = PCB149 (ng/g)
	LBX087 = PCB87 (ng/g)
	URXUPT = Platinum, urine (ng/mL)
	LBDBANO = Basophils number
	DRD370MQ = # of times salmon eaten in past 30 days
	DRD370AQ = # of times breaded fish produ

In [6]:
for i in range(len(dfs)):
    var_types   = cl.describe.get_types(dfs[i])
    var_unknown = var_types[var_types == 'unknown'].index
    dfs[i] = cl.modify.make_continuous(dfs[i], only=var_unknown)

Running make_continuous
--------------------------------------------------------------------------------
Set 16 of 390 variable(s) as continuous, each with 4,724 observations
Running make_continuous
--------------------------------------------------------------------------------
Set 14 of 390 variable(s) as continuous, each with 4,339 observations
Running make_continuous
--------------------------------------------------------------------------------
Set 10 of 390 variable(s) as continuous, each with 5,123 observations
Running make_continuous
--------------------------------------------------------------------------------
Set 9 of 390 variable(s) as continuous, each with 4,751 observations


In [7]:
for i in range(len(dfs)):
    cl.describe.summarize(dfs[i])

4,724 observations of 390 variables
	44 Binary Variables
	5 Categorical Variables
	341 Continuous Variables
	0 Unknown-Type Variables

4,339 observations of 390 variables
	44 Binary Variables
	5 Categorical Variables
	341 Continuous Variables
	0 Unknown-Type Variables

5,123 observations of 390 variables
	44 Binary Variables
	5 Categorical Variables
	341 Continuous Variables
	0 Unknown-Type Variables

4,751 observations of 390 variables
	44 Binary Variables
	5 Categorical Variables
	341 Continuous Variables
	0 Unknown-Type Variables



## Survey weights

Much information on the issue on survey weights can be found [here](https://wwwn.cdc.gov/nchs/nhanes/tutorials/module3.aspx)

In [8]:
os.chdir(datapath)
survey_design_discovery = pd.read_csv("weights_discovery.txt", sep="\t")\
                            .rename(columns={'SEQN':'ID'})\
                            .set_index("ID")
survey_design_discovery.head()

,SDDSRVYR,SDMVPSU,SDMVSTRA,WTINT2YR,WTINT4YR,WTMEC2YR,WTMEC4YR,WTDRD1,WTDR4YR,WTSBA2YR,...,WTSPH2YR,WTSAU4YR,WTSAF4YR,WTSAF2YR,WTSCI4YR,WTSPH4YR,WTSCI2YR,WTSVOC2Y,WTSAU2YR,WTUIO2YR
ID,,,,,,,,,,,,,,,,,,,,,
1,1,1,5,9727.078709,4291.490177,10982.898896,4456.206594,14809.893854,6066.128663,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1,3,1,26678.636376,14203.335998,28325.384898,15336.199717,26899.708892,14921.934292,NaN,...,NaN,NaN,33073.267573,60586.147294,NaN,NaN,NaN,NaN,NaN,NaN
3,1,2,7,43621.680548,20123.763507,46192.256945,21258.466625,34388.840334,15866.447640,NaN,...,142734.07396,NaN,52434.225472,121969.841152,NaN,69163.765317,NaN,NaN,NaN,NaN
4,1,1,2,10346.119327,4582.132308,10251.260020,4562.389068,7159.829389,3218.099911,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,1,2,8,91050.846620,44161.868201,99445.065735,45985.967832,127746.359176,58973.611131,225514.73187,...,NaN,NaN,98468.806492,234895.205650,NaN,NaN,NaN,NaN,NaN,NaN


In [9]:
os.chdir(datapath)
survey_design_replication = pd.read_csv("weights_replication.txt", sep="\t")\
                            .rename(columns={'SEQN':'ID'})\
                            .set_index("ID")
survey_design_replication.head()

,SDDSRVYR,SDMVPSU,SDMVSTRA,WTINT2YR,WTMEC2YR,WTS_FFQ,BMXWT,BMIWT,CVDWTIM,WTSVOC2Y,...,WTSC2YR,WTSAF2YR,WTSA2YR,WTSCI2YR,WTSAU2YR,WTAL2YR,LBXDWT,WTSOG2YR,WTSC2YRA,WTSPC2YR
ID,,,,,,,,,,,,,,,,,,,,,
21005,3,2,39,5512.320949,5824.782465,4018.029553,137.6,NaN,NaN,NaN,...,18945.492755,14084.2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
21006,3,1,41,5422.140453,5564.039715,3838.164968,55.2,NaN,2.0,NaN,...,NaN,13081.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
21007,3,2,35,39764.177412,40591.066325,26635.551187,47.9,NaN,2.0,NaN,...,NaN,NaN,132847.074292,NaN,NaN,NaN,NaN,NaN,NaN,NaN
21008,3,1,32,5599.499351,5696.750596,NaN,70.0,NaN,NaN,NaN,...,18529.060575,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
21009,3,2,31,97593.678977,97731.727244,67060.189883,103.1,NaN,NaN,182899.246183,...,276204.298055,NaN,NaN,NaN,186554.215404,NaN,NaN,NaN,NaN,NaN


In [10]:
# Divide replication weights by 2 to get 4 year weights
survey_design_replication.iloc[:,3:] = survey_design_replication.iloc[:,3:] / 2
survey_design_replication.head()

,SDDSRVYR,SDMVPSU,SDMVSTRA,WTINT2YR,WTMEC2YR,WTS_FFQ,BMXWT,BMIWT,CVDWTIM,WTSVOC2Y,...,WTSC2YR,WTSAF2YR,WTSA2YR,WTSCI2YR,WTSAU2YR,WTAL2YR,LBXDWT,WTSOG2YR,WTSC2YRA,WTSPC2YR
ID,,,,,,,,,,,,,,,,,,,,,
21005,3,2,39,2756.160474,2912.391233,2009.014777,68.80,NaN,NaN,NaN,...,9472.746377,7042.1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
21006,3,1,41,2711.070226,2782.019857,1919.082484,27.60,NaN,1.0,NaN,...,NaN,6540.5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
21007,3,2,35,19882.088706,20295.533162,13317.775593,23.95,NaN,1.0,NaN,...,NaN,NaN,66423.537146,NaN,NaN,NaN,NaN,NaN,NaN,NaN
21008,3,1,32,2799.749676,2848.375298,NaN,35.00,NaN,NaN,NaN,...,9264.530288,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
21009,3,2,31,48796.839489,48865.863622,33530.094942,51.55,NaN,NaN,91449.623091,...,138102.149027,NaN,NaN,NaN,93277.107702,NaN,NaN,NaN,NaN,NaN


In [11]:
os.chdir(datapath)
# These files map variables to their correct weights, and were compiled by reading throught the NHANES codebook
var_weights = pd.read_csv("VarWeights.csv")
var_weights.head()

,variable_name,discovery,replication
0,99999,WTMEC4YR,WTMEC2YR
1,ACETAMINOPHEN__CODEINE,WTMEC4YR,WTMEC2YR
2,ACETAMINOPHEN__CODEINE_PHOSPHATE,WTMEC4YR,WTMEC2YR
3,ACETAMINOPHEN__HYDROCODONE,WTMEC4YR,WTMEC2YR
4,ACETAMINOPHEN__HYDROCODONE_BITARTRATE,WTMEC4YR,WTMEC2YR


In [12]:
# Convert the data to two dictionaries for convenience
weights_discovery   = var_weights.set_index('variable_name')['discovery'].to_dict()
weights_replication = var_weights.set_index('variable_name')['replication'].to_dict()

# Remove the list of variables from replication (trows an error and it's not in the dataset either)
#remove_rep_dict = ['LBXTSH', 'URX24D', 'URX25T', 'URX4FP', 'URXACE', 'URXATZ', 'URXCB3', 'URXCCC', 'URXCMH', 'URXCPM',
#                   'URXDEE', 'URXDIZ', 'URXDPY', 'URXMET', 'URXOPM']
#for i in remove_rep_dict:
#    weights_replication.pop(i, None)

In [13]:
cols = survey_design_replication.columns
vals = set(weights_replication.values())

#Which survey weights are not present
weights_absent = []
for i in set(vals):
    if i not in cols:
        print(i + ' is not in cols')
        weights_absent.append(i)

#Which variables need those survey weights
remove_vars = []
for key, value in weights_replication.items():
    for i in weights_absent:
        if i == value: 
            remove_vars.append(key)
            
# Remove those vars from the weights dict
for i in remove_vars:
    weights_discovery.pop(i)
    weights_replication.pop(i)

WTSTH2YR is not in cols
WTSPH2YR is not in cols
WTSPP2YR is not in cols


## Define phenotypes and covariates


In [14]:
phenotype = ["LBXSCRINV","URXUCR","LBXSCR","LBXSATSI","LBXSAL","URXUMASI","URXUMA","LBXSAPSI","LBXSASSI","LBXSC3SI",
             "LBXSBU","LBXBAP","LBXCPSI","LBXCRP","LBXSCLSI","LBXSCH","LBDHDL","LBDLDL","LBXSGTSI","LBXSGB",
             "LBXGLU","LBXGH","LBXHCY","LBXSIR","LBXSLDSI","LBXMMA","LBXSOSSI","LBXSPH","LBXSKSI",	
             "LBXSNASI","LBXSTB","LBXSCA","LBXSTP","LBXSTR","LBXSUA","LBDBANO","LBXBAPCT",
             "LBDEONO","LBXEOPCT","LBXHCT","LBXHGB","LBDLYMNO","LBXMCHSI","LBXLYPCT","LBXMCVSI","LBXMPSI","LBDMONO",
             "LBXMOPCT","LBXPLTSI","LBXRBCSI","LBXRDW","LBDNENO","LBXNEPCT"] # I removed the ones that were deleted in the QC process
for v in phenotype:
    if v in var_descr_dict:
        print(f"\t{v} = {var_descr_dict[v]}")
        
covariates = ["black", "mexican", "other_hispanic", "other_eth", "SES_LEVEL", "RIDAGEYR", "SDDSRVYR","BMXBMI"]

	LBXSCRINV = 1/Creatinine (mg/dL)
	URXUCR = Creatinine, urine (mg/dL)
	LBXSCR = Creatinine (mg/dL)
	LBXSATSI = ALT (U/L)
	LBXSAL = Albumin (g/dL)
	URXUMASI = Albumin, urine (mg/L) SI
	URXUMA = Albumin, urine (ug/mL)
	LBXSAPSI = Alkaline phosphotase (U/L)
	LBXSASSI = AST (U/L)
	LBXSC3SI = Bicarbonate (mmol/L)
	LBXSBU = Blood urea nitrogen (mg/dL)
	LBXBAP = Bone alkaline phosphotase (ug/L)
	LBXCPSI = C-peptide: SI(nmol/L)
	LBXCRP = C-reactive protein(mg/dL)
	LBXSCLSI = Chloride (mmol/L)
	LBXSCH = Cholesterol, total (mg/dL)
	LBDHDL = Direct HDL-Cholesterol (mg/dL)
	LBDLDL = LDL-cholesterol (mg/dL)
	LBXSGTSI = GGT (U/L)
	LBXSGB = Globulin (g/dL)
	LBXGLU = Glucose, plasma (mg/dL)
	LBXGH = Glycohemoglobin (%)
	LBXHCY = Homocysteine (umol/L)
	LBXSIR = Iron (ug/dL)
	LBXSLDSI = LDH (U/L)
	LBXMMA = Methylmalonic acid (umol/L)
	LBXSOSSI = Osmolality (mOsml/L)
	LBXSPH = Phosphorus (mg/dL)
	LBXSKSI = Potassium (mmol/L)
	LBXSNASI = Sodium (mmol/L)
	LBXSTB = Bilirubin, total (mg/dL)
	LBXSCA = Calcium

In [15]:
# Create 4 survey designs
surveys = [survey_design_discovery.loc[dfs[0].index], survey_design_discovery.loc[dfs[1].index],
          survey_design_replication.loc[dfs[2].index], survey_design_replication.loc[dfs[3].index]]
sds = []
for i in range(len(dfs)):
    if i == 0 or i == 1:
        w = weights_discovery
    else:
        w = weights_replication
        
    sds.append(cl.survey.SurveyDesignSpec(survey_df=surveys[i],
                                                   strata="SDMVSTRA",
                                                   cluster="SDMVPSU",
                                                   nest=True,
                                                   weights=w, # This are the same in discovery and replication
                                                   single_cluster='adjust'))

In [16]:
#Omit pandas warning
import warnings
warnings.filterwarnings("ignore")

#Run analysis
results_d_f = []
results_d_m = []
results_r_f = []
results_r_m = []
for i in range(len(dfs)):
    for current_pheno in phenotype:
        print(current_pheno)
        drop_phenos = [p for p in phenotype if p != current_pheno]
        ewas_result_dis = cl.analyze.ewas(phenotype = current_pheno,
                                          covariates = covariates,
                                          min_n = 200,
                                          data = dfs[i].drop(columns=drop_phenos),
                                          regression_kind = 'weighted_glm',
                                          survey_design_spec=sds[i])
        if i == 0:
            results_d_f.append(ewas_result_dis)
        elif i == 1:
            results_d_m.append(ewas_result_dis)
        elif i == 2:
            results_r_f.append(ewas_result_dis)
        elif i == 3:
            results_r_m.append(ewas_result_dis)
                
    
results = [pd.concat(results_d_f).merge(pd.concat(results_r_f), how = 'inner', 
                             left_index=True, right_index=True, suffixes =['_d', '_r'] ), 
           pd.concat(results_d_m).merge(pd.concat(results_r_m), how = 'inner', 
                             left_index=True, right_index=True, suffixes =['_d', '_r'] )]

LBXSCRINV
WeightedGLMRegression
-------------------------
Continuous Outcome (family = Gaussian): 'LBXSCRINV'
Using 4,459 of 4,724 observations
	265 are missing a value for the outcome variable
Regressing 329 variables
	39 binary variables
	4 categorical variables
	286 continuous variables
Survey Design
	4,724 rows in the survey design data
	Strata: 28 unique values of SDMVSTRA
	Cluster: 57 unique values of SDMVPSU (nested)
	FPC: None
	Multiple Weights: 9 unique weights associated with 1,066 variables
	Drop Unweighted: False
	Single Cluster ('Lonely PSU') Option: adjust
	Subsets: 0 applied
		Keeping 4,724 of 4,724 observations
-------------------------
Running 39 binary variables...
	Finished Running 39 binary variables
Running 4 categorical variables...
	Finished Running 4 categorical variables
Running 286 continuous variables...
	Finished Running 286 continuous variables
62 regression variables had an error
	DRD370T = NULL due to: 3 observations are missing weights when the variable 

## Meta-analyze

In [17]:
def meta_analysis(betas, ses):
    '''
    Perform a meta-analysis from a given set of beta and se coefficients
    '''
    import numpy as np
    import scipy.stats as stats
    
    w1 = 1 / np.power(ses[0],2)
    w2 = 1 / np.power(ses[1],2)
    meta_se   = np.sqrt( 1 / (w1 + w2))
    meta_beta = (betas[0] * w1 + betas[1] * w2) / (w1 + w2)
    zeta = meta_beta / meta_se
    pval = 2 * stats.norm.cdf(-abs(zeta))
    
    return[meta_se, meta_beta, pval]

In [18]:
# Males vs females meta-analysis
drop_cols = ['Variable_type_r', 'Weight_r', 'Converged_r', 'LRT_pvalue_r', 'Diff_AIC_r'] #Columns to drop

for i in range(len(results)): # Calculate the meta-analysis parameters
    vals = meta_analysis([results[i]['Beta_d'], results[i]['Beta_r']] , [results[i]['SE_d'], results[i]['SE_r']])
    results[i]['SE']     = vals[0]
    results[i]['Beta']   = vals[1]
    results[i]['pvalue'] = vals[2]
    results[i]['Variable_pvalue'] = vals[2]
    results[i]['N'] = results[i]['N_d'] + results[i]['N_r']
    results[i] = results[i].rename(columns={'Variable_type_d':'Variable_type',
                                            'Weight_d': 'Weight',
                                            'Converged_d':'Converged',
                                            'LRT_pvalue_d':'LRT_pvalue',
                                            'Diff_AIC_d':'Diff_AIC'})
    # Remove some unnecesary columns
    results[i] = results[i].drop(drop_cols, axis=1)

In [19]:
# Final meta analysis (males with females)
vals = meta_analysis([results[0]['Beta'], results[1]['Beta'] ], [results[0]['SE'], results[1]['SE']] )

final_meta = pd.DataFrame(results[0].iloc[:,0:3], index=results[0].index)
final_meta['N'] = results[0]['N'] + results[1]['N']
final_meta['LRT_pvalue'] = results[0]['LRT_pvalue']
final_meta['Diff_AIC']   = results[0]['Diff_AIC']
final_meta['SE']     = vals[0]
final_meta['Beta']   = vals[1]
final_meta['pvalue'] = vals[2]
final_meta['Variable_pvalue'] = vals[2]
cl.analyze.add_corrected_pvalues(final_meta)

## Sex difference analysis

There are two complementary analysis done here:
- First, a wide difference test (better suited to detect opposite effects), that will take all possible tests, calculate bonferroni corrected pvalues, and then only keep those that have opposite effects between sexes
- Second, a filtering first test (better suited to detect effects that are only or more pronounced in one sex), that will take the overall meta-analyzed association, filtering by (what threshold?), and then test for differences between sexes and retain only those that are not opposite effects

In [20]:
# Merge datasets and rename and remove columns
drop_cols   = ['Variable_type_m', 'Weight_m', 'Converged_m', 'LRT_pvalue_m', 'Diff_AIC_m',
               'Variable_type_f', 'Weight_f', 'Converged_f', 'LRT_pvalue_f', 'Diff_AIC_f'] #Columns to drop

merge_sex = pd.merge(results[0], results[1], how = 'inner', left_index=True, right_index=True, suffixes =['_f', '_m'])

# Kepp only those converged in both males and females
keep_results = (merge_sex['Converged_f'] & merge_sex['Converged_m'])
merge_sex    = merge_sex.loc[keep_results]

# Remove and rename columns
merge_sex = merge_sex.drop(drop_cols, axis=1)

# Merge with overall meta analysis
final_result = pd.merge(final_meta, merge_sex, how = 'inner', left_index=True, right_index=True, suffixes=[None, 'y'])

In [36]:
# Estimate differences
t1 = np.array(final_result['Beta_f']) - np.array(final_result['Beta_m'])
t2 = np.sqrt(np.power(np.array(final_result['SE_f']), 2) + np.power(np.array(final_result['SE_m']), 2))
zdiff   = t1 / t2
pval = 2*stats.norm.cdf(-abs(zdiff))
final_result['pvalue_diff'] = pval
final_result['SE_diff'] = t2
final_result['Beta_diff'] = t1
final_result['Variable_pvalue_diff'] = pval

# Get corrected pvalues
from statsmodels.stats.multitest import multipletests
final_result.loc[~final_result['pvalue_diff'].isna(), 'pvalue_diff_bonferroni'] = multipletests(
                                                                                        final_result.loc[~final_result['pvalue_diff'].isna(), 'pvalue_diff'], 
                                                                                        method='bonferroni'
                                                                                        )[1]

# Save file
os.chdir(respath)
final_result.to_csv('Difference_test.csv')